# 02 · 프롬프트 설계
**블록 3 · 프롬프트 설계** — 실습 60분

여기서 만드는 v1이 오후 블록 4에서 20문항으로 채점받는 **1차**의 재료입니다. 지금은 그 전에 "쓸 만한 프롬프트 하나"를 준비하는 단계입니다.

사내 문의 메시지 하나를 받아 **카테고리 · 긴급도 · 요약**으로 분류하는 프롬프트를 씁니다.

```json
{"카테고리": "IT", "긴급도": "보통", "요약": "비밀번호 재설정 요청"}
```

- 카테고리 — 비품 · 시설 · 인사 · IT · 기타 중 하나
- 긴급도 — 높음 · 보통 · 낮음 중 하나

**통과 기준** — 10번 돌려 10번 모두 ① JSON으로 읽히고 ② 카테고리·긴급도가 위 보기 안에 있고 ③ 요약이 비어 있지 않음. 아래 `try_prompt()`가 이 세 가지를 그대로 세어 "통과 x / n"으로 보여줍니다.

| 순서 | 할 일 | 시간 |
|---|---|---|
| 1 | 문의 고르기 — 짧은 문의 하나 준비 | 5분 |
| 2 | v0 — 한 줄짜리 지시로 먼저 돌려 보고 무엇이 부족한지 적는다 | 10분 |
| 3 | v1 — 프롬프트 파일을 한 줄씩 채우고, 채울 때마다 돌려 본다 | 30분 |
| 4 | 짝 점검 체크리스트 | 15분 |

In [ ]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # day 폴더로 이동 — data/ · prompts/ · src/ 가 여기 있다

from common import llm      # 공용 API 도구 (ETRI_LLM_3days/common/llm.py)
print("모델:", llm.MODEL, "| 모의 모드" if llm.MOCK else "| 실제 호출")
import json
from src.grade import build, parse, schema_errors

## 1. 문의 고르기

In [ ]:
doc = "탕비실 정수기가 고장나서 물이 안 나옵니다. 교체 부탁드립니다."   # ✏️ 내 업무 문의로 바꿔도 됩니다
print(doc)

## 2. v0 — 한 줄짜리 지시
`prompts/prompt_v0.txt`를 그대로 돌립니다. 결과를 보고 **무엇이 부족한지** 적습니다.

In [ ]:
def try_prompt(prompt_text, document, n=1, show=True):
    """프롬프트를 n번 돌려, 도입부의 통과 기준 세 가지를 모두 만족한 횟수를 센다.
    ① JSON으로 읽히는가  ② 카테고리·긴급도가 보기 안인가  ③ 요약이 비어 있지 않은가"""
    system, user = build(prompt_text, document)
    if show:
        print(f"=== system에 들어간 것 ===\n{system}\n")
        print(f"=== user에 들어간 것 ===\n{user}\n")
    ok = 0
    for i in range(n):
        r = llm.call(user, system=system, max_tokens=500)
        try:
            out = parse(r.text)                                   # ①
            errs = schema_errors(out)                             # ② ③
            if errs:
                note = "JSON은 읽힘 · 보기 밖 — " + errs[0]
            else:
                ok += 1
                note = f"통과 · 카테고리={out['카테고리']} · 긴급도={out['긴급도']}"
        except ValueError as e:
            note = f"JSON 실패 — {e}"
        if show:
            print(f"--- {i+1}회 · {note} · 출력 {r.output_tokens} 토큰")
            print(r.text.strip()[:600])
    print(f"\n통과 {ok} / {n}   (①JSON ②보기 안 ③요약 있음 — 셋 다 만족한 횟수)")
    return ok

v0 = llm.load_prompt("prompts/prompt_v0.txt")
print(v0, "\n========")
try_prompt(v0, doc)

**v0에서 부족했던 것** (한 줄씩). 위 출력에서 이 세 가지를 확인해 적습니다:
- 형식 — `<json>` 태그 안에 들어 있나, 아니면 코드블록(```)이나 설명이 붙어 있나? →
- 키 이름 — `카테고리`·`긴급도`·`요약` 세 개인가, 아니면 모델이 다른 이름을 지어냈나? →
- 값 — 카테고리가 다섯 보기 중 하나인가, 긴급도가 세 보기 중 하나인가? → 

## 3. v1 — 아래 표를 한 줄씩 채우고 바로 돌려본다
`prompts/prompt_v1.txt`를 엽니다. 파일 안에 **`TODO:`로 시작하는 줄이 네 개** 있습니다. `TODO`는 "여기는 네가 써야 할 자리"라는 표시일 뿐이고, 할 일은 **그 줄을 통째로 지우고 그 자리에 내 문장을 쓰는 것**입니다. 다 채우면 파일에 `TODO`라는 글자가 하나도 남지 않아야 합니다 — 아래 셀이 그걸로 "아직 안 채웠다"를 판단합니다.

예를 들어 `<역할>`은 이렇게 바뀝니다:
```
전:  <역할>
     TODO: 누가, 무엇을 위해 이 일을 하는지 한두 줄 (예: 총무팀 헬프데스크 담당자)
     </역할>

후:  <역할>
     너는 총무팀 헬프데스크 담당자다. 직원이 보낸 문의를 읽고 분류한다.
     </역할>
```

표의 항목을 **위에서부터 한 줄씩** 채우고, 한 줄 채울 때마다 바로 아래 셀을 다시 실행해 뭐가 달라지는지 봅니다. 다 채운 뒤에 한꺼번에 실행하지 않습니다 — 그러면 뭐가 효과 있었는지 알 수 없습니다.

| 채울 내용 | 넣는 곳 |
|---|---|
| 역할을 구체적으로 — 누가, 무엇을 위해 | `<역할>` |
| 할 일을 구체적으로 — 카테고리 다섯 개 · 긴급도 세 개를 각각 뭘 기준으로 고르는지 | `<할 일>` |
| "~하지 마" 대신 "~해라"로 쓰기 | `<역할>` `<할 일>` 전체 |
| 예시 하나 — 짧은 문의와 정답 JSON | `<예시>` |
| 원하는 JSON 모양을 그대로 보여주기 | `<출력 형식>` |

파일 맨 위 `=== system ===` 아래는 system으로, `=== user ===` 아래는 user로 보냅니다. `{document}` 자리에 문의가 들어갑니다.

### 먼저 전체 과정을 한 번 구경한다 — TODO를 하나씩 채우면 뭐가 바뀌나
내 파일을 고치기 전에, 아래 셀이 **TODO 템플릿에서 출발해 TODO 네 개를 하나씩 채우면서 매번 3번씩 돌려** 봅니다. 파일은 건드리지 않고 메모리에서만 채웁니다. 단계마다 통과 수와 함께 **모델이 낸 (카테고리, 긴급도) 값 세 개**를 그대로 찍어서, 뭐가 바뀌고 뭐가 안 바뀌는지 보이게 합니다.

`<출력 형식>`을 **먼저** 채웁니다 — 이게 안 되면 파싱이 막혀 나머지 셋이 뭘 했는지 볼 수 없기 때문입니다.

In [ ]:
import re

def fill(text, tag, body):
    """<tag>…</tag> 안쪽을 body로 바꾼다 (TODO 줄을 지우고 내 문장을 쓰는 것과 같다)"""
    return re.sub(rf"<{tag}>.*?</{tag}>", f"<{tag}>\n{body}\n</{tag}>", text, flags=re.S)

steps = [
    ("출력 형식", "설명이나 코드블록(```) 없이, 아래처럼 <json></json> 태그 안에만 써라. 그 외에는 아무것도 쓰지 않는다.\n"
                  "<json>{\"카테고리\": \"…\", \"긴급도\": \"…\", \"요약\": \"…\"}</json>"),
    ("역할",      "너는 총무팀 헬프데스크 담당자다. 직원이 보낸 문의 메시지를 읽고 분류한다."),
    ("할 일",     "<문의>를 아래 기준으로 분류하라.\n"
                  "- 카테고리: 비품(소모품·책상·의자 등 구매 요청) · 시설(냉난방·조명·주차장·건물 설비) · "
                  "인사(연차·급여·휴가) · IT(컴퓨터·네트워크·계정·장비 고장) · 기타(위 네 가지에 뚜렷이 안 맞을 때)\n"
                  "- 긴급도: 높음 · 보통 · 낮음\n- 요약: 한 줄"),
    ("예시",      "문의: \"에어컨 필터 청소가 필요합니다.\"\n"
                  "정답: <json>{\"카테고리\": \"시설\", \"긴급도\": \"낮음\", \"요약\": \"에어컨 필터 청소 요청\"}</json>"),
]

def run3(text):
    """3번 돌려 (통과 수, 모델이 낸 (카테고리, 긴급도) 값들) 을 돌려준다."""
    system, user = build(text, doc)
    ok, vals = 0, []
    for _ in range(3):
        t = llm.call(user, system=system, max_tokens=300).text
        try:
            out = parse(t)
            if not schema_errors(out): ok += 1
            vals.append((out.get("카테고리"), out.get("긴급도")))
        except ValueError:
            vals.append("파싱 실패")
    return ok, vals

cur = llm.load_prompt("prompts/prompt_v1.txt")      # TODO 템플릿 (파일은 안 바꾼다)
ok, vals = run3(cur)
print(f"0) TODO 템플릿 그대로        통과 {ok}/3   {vals}")
for i, (tag, body) in enumerate(steps, 1):
    cur = fill(cur, tag, body)
    ok, vals = run3(cur)
    print(f"{i}) <{tag}> 채움{' '*(10-len(tag))}통과 {ok}/3   {vals}")

읽는 법:
- **`<출력 형식>` 하나로 통과가 0 → 3이 됩니다.** "`<json></json>` 안에만, 코드블록 없이"라는 지시가 ①②③을 전부 결정합니다. 그 전엔 모델이 JSON을 ` ``` `로 감싸서 첫 글자부터 실패했습니다.
- **`<역할>`·`<할 일>`·`<예시>`는 통과 수를 안 바꿉니다.** 바꾸는 건 **값**입니다 — 특히 긴급도가 `<할 일>`·`<예시>` 뒤에 움직이는지 보세요. 문서 한 건으로는 그 움직임이 **맞는 방향인지 알 수 없습니다.** 그걸 알려면 정답이 있는 문항 여러 개로 재야 하고, 그게 오후 03 노트북입니다.
- 거기서 효과가 없는 것으로 나오는 항목은 **빼도 됩니다.** "기법을 넣었으니 좋아졌겠지"가 아니라 재서 확인하는 게 이 수업의 요지입니다.

이제 **내 파일**로 같은 일을 합니다.

In [ ]:
V1_PATH = "prompts/prompt_v1.txt"      # ✏️ 내 것 대신 풀이본을 보려면 "solutions/prompts/prompt_v1.txt"
v1 = llm.load_prompt(V1_PATH)           # 파일을 고친 뒤 이 셀을 다시 실행

if "TODO" in v1:
    print("=" * 70)
    print("⚠️  prompt_v1.txt 에 'TODO:' 로 시작하는 줄이 아직 남아 있습니다.")
    print("    그 줄은 자리 표시입니다 — 줄을 지우고 그 자리에 내 문장을 쓰세요 (3절의 전/후 예시 참고).")
    print("    채우기 전까지는 이 셀부터 아래 셀 전부가 '실패'로 나옵니다 — 그게 정상입니다.")
    print("=" * 70, "\n")

try_prompt(v1, doc)

> 막히면 `solutions/prompts/prompt_v1.txt`를 열어 봅니다. 그대로 베끼기보다, 내 v1과 **무엇이 다른지** 비교합니다.

## 4. 짝 점검 체크리스트
v1을 짝과 바꿔 보고, 아래 다섯 가지를 하나씩 확인합니다. 하나라도 "아니오"면 3절로 돌아가 고칩니다.

- [ ] **형식** — 10번 돌려 10번 다 `json.loads`로 읽히는가? (코드블록·설명 없이)
- [ ] **보기 안** — 카테고리가 항상 다섯 보기(비품·시설·인사·IT·기타) 중 하나인가?
- [ ] **보기 안** — 긴급도가 항상 세 보기(높음·보통·낮음) 중 하나인가?
- [ ] **요약 있음** — 요약이 비어 있지 않은가?
- [ ] **구조** — `=== system ===`에는 역할·규칙·형식만, `=== user ===`에는 문의만 들어가 있는가?

아래 셀이 위 **네 개를 자동으로** 셉니다(`try_prompt`가 세는 통과 기준이 바로 이것). 다섯 번째(구조)만 파일을 열어 눈으로 봅니다.

In [ ]:
ok = try_prompt(v1, doc, n=10, show=False)
print("통과" if ok == 10 else "아직 — 형식 지시 · 예시를 다시 봅니다")

## 5. 생각할 자리를 준다
**방금 만든 v1에, 한 가지를 더합니다.** 바로 분류하게 하지 말고, **판단 근거를 먼저 한 줄 적게** 합니다. 출력 형식은 v1 것을 그대로 쓰고, 규칙만 하나 추가합니다.

In [ ]:
import re

system, user = build(v1, doc)      # v1의 <출력 형식>을 그대로 쓴다
EVIDENCE = ("\n\n<근거 규칙>\n<json>을 쓰기 전에 <근거>에 이 문의를 그렇게 분류한 이유를 "
            "한 문장으로 적어라.\n</근거 규칙>")

r = llm.call(user, system=(system or "") + EVIDENCE, max_tokens=500)
print(r.text.strip())

m = re.search(r"<근거>(.*?)</근거>", r.text, re.S)
print("\n→ 코드가 디버깅용으로 꺼낸 <근거>:", (m.group(1).strip() if m else "(없음)"))
try:
    print("→ parse() 결과(<json>만 읽음):", parse(r.text))
except ValueError as e:
    print("→ parse() 실패:", e)
    print("   v1의 <출력 형식>에 '<json></json> 안에만 써라'가 들어 있는지 확인하세요 — 3절로 돌아갑니다.")

## 6. 한 덩어리 일을 세 단계로 나눈다 — 분류하기(모델) → 검사하기(코드) → 답장 쓰기(모델)
이 절은 프롬프트를 잘 쓰는 기술이 아니라 **작업을 어떻게 나눌지**에 대한 얘기입니다. 프롬프트로 해결하려 하지 말고 코드로 해야 하는 부분이 있다는 것을 봅니다.

"문의를 분류하고, 분류가 정해진 보기 안에 있는지 확인하고, 직원에게 보낼 접수 답장을 써 줘"를 한 번에 시킬 수도 있습니다. 그러면 모델이 보기에 없는 카테고리를 만들어내도 그냥 넘어갈 수 있습니다.

아래 셀은 이 일을 셋으로 나눕니다.
- **① 분류** — v1으로 카테고리·긴급도·요약을 뽑습니다 (모델)
- **② 검사** — 카테고리·긴급도가 정해진 보기 안에 있는지 봅니다 (**코드** — 정해진 보기 밖의 말을 걸러내는 건 모델보다 코드가 확실합니다)
- **③ 답장** — ②를 통과한 분류만 보고 접수 답장을 씁니다 (모델). ②에서 걸리면 ③은 건너뛰고 ①로 돌아갑니다.

In [ ]:
# ① 분류 — 지금까지 쓴 v1을 그대로 쓴다
system, user = build(v1, doc)
r1 = llm.call(user, system=system)
try:
    data = parse(r1.text)
except ValueError as e:
    data = None
    print("① 분류 실패:", e)
print("① 분류:", json.dumps(data, ensure_ascii=False))

# ② 검사 — 모델이 아니라 코드가 본다
from src.grade import CATEGORIES, URGENCY
problems = []
if data is None:
    problems.append("JSON 아님")
else:
    if data.get("카테고리") not in CATEGORIES:
        problems.append(f"카테고리 '{data.get('카테고리')}'가 보기 밖")
    if data.get("긴급도") not in URGENCY:
        problems.append(f"긴급도 '{data.get('긴급도')}'가 보기 밖")
    if not data.get("요약"):
        problems.append("요약 없음")
print("② 검사:", problems or "통과")

# ③ 답장 — 검사를 통과한 분류만 보고 접수 답장 작성
if not problems:
    r3 = llm.call(
        "아래 분류 결과만 보고 문의자에게 보낼 접수 답장을 두 줄로 쓰세요. 분류에 없는 내용은 쓰지 않습니다.\n\n"
        + json.dumps(data, ensure_ascii=False),
        max_tokens=300)
    print("③ 답장:\n" + r3.text.strip())
else:
    print("③ 답장: 건너뜀 — ①로 돌아가 고칩니다")

## 7. 일반화된 프롬프트를 여러 다른 문의에 적용한다
v1은 문의 한 건에 맞춘 프롬프트가 아닙니다. 역할 · 할 일 · 출력 형식 · 예시는 고정하고 `{document}` 자리만 비워 둔 **일반화된 프롬프트**입니다.

아래 셀은 그 v1을 `data/tests.jsonl`에 있는 문의 5건에 차례로 적용해, **프롬프트를 고치지 않고도 다른 문의에서 같은 모양의 결과가 나오는지** 확인합니다. 오후 블록 4는 같은 방법으로 문의 20건에 적용합니다. (이 셀은 아무것도 새로 저장하지 않습니다.)

In [ ]:
from src.grade import load_tests

for t in load_tests()[:5]:
    system, user = build(v1, t["input"])
    r = llm.call(user, system=system)
    try:
        out = parse(r.text)
        print(f"#{t['id']:>2}  {t['input'][:24]:<24} → {out.get('카테고리')} · {out.get('긴급도')}")
    except ValueError as e:
        print(f"#{t['id']:>2}  실패 {e}")

## 제출할 것
- `prompts/prompt_v0.txt` · `prompts/prompt_v1.txt` — 두 버전을 모두 남긴다
- v0에서 v1로 넘어오며 추가한 것과 그 이유 세 줄:
  1.
  2.
  3. 